# PCG-to-ECG Signal Reconstruction: Cross-Dataset Validation

**Ablation Study Code** for: *PCG-to-ECG Signal Reconstruction Using Conditional GAN with Multi-Scale Attention* (EUSIPCO 2026)

**Authors:** Zafar Ahmad, Nazia Perwaiz, Muhammad Salman Khan

---

This notebook reproduces the cross-dataset validation from the ablation study.
We test the Trail-4 model (trained on EPHNOGRAM) on the SensSmart-Tech database without retraining.

**Dataset:** SensSmart-Tech (32 subjects, 338 recordings, 1 kHz resampled to 8 kHz)

**Results (310 records, post-rescaling):**

| Metric | EPHNOGRAM | SmartTech | Delta |
|--------|-----------|-----------|-------|
| PSNR (dB) | 17.10 | 17.04 | -0.4% |
| SSIM | 0.903 | 0.742 | -17.9% |
| Correlation | 0.610 | -0.003 | -100.4% |
| R-peak Err | 0.126 | 1.400 | +1011% |

PSNR matches across datasets. Correlation collapses due to domain shift.

## Imports

In [ ]:
import numpy as np
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.signal import resample_poly, butter, filtfilt
from scipy.stats import pearsonr
from skimage.metrics import structural_similarity as compare_ssim
import scipy.signal as signal

try:
    import wfdb
except ImportError:
    print("Installing wfdb...")
    import subprocess
    subprocess.check_call(["pip", "install", "wfdb"])
    import wfdb

---
## Configuration

Paths assume Google Colab with Drive mounted.
Adjust `CONFIG` for local runs.

In [ ]:
CONFIG = {
    # Data paths (Google Colab / Drive)
    'senssmarttech_dir': '/content/drive/MyDrive/October Thesis Dataset/senssmarttech-database',
    'wfdb_dir': '/content/drive/MyDrive/October Thesis Dataset/senssmarttech-database/WFDB',
    'preprocessed_dir': '/content/drive/MyDrive/October Thesis Dataset/milestone2_preprocessed',
    'model_path': '/content/drive/MyDrive/October Thesis Dataset/trail4_generator.pth',
    'results_dir': '/content/drive/MyDrive/October Thesis Dataset/crossdataset_results',

    # Signal parameters
    'original_fs': 1000,       # SensSmart-Tech native sampling rate
    'target_fs': 8000,         # EPHNOGRAM / model sampling rate
    'segment_length': 64000,   # 8 seconds at 8 kHz

    # Preprocessing
    'ecg_bandpass': [0.5, 40],   # ECG filter band (Hz)
    'pcg_bandpass': [25, 400],   # PCG filter band (Hz)
    'filter_order': 4,           # Butterworth order
    'fix_polarity': True,
    'ecg_lead': 'II',

    # Quality thresholds
    'min_snr_pcg': 0,
    'max_kurtosis': 100,
    'min_zcr': 0.0001,

    # Inference
    'scenario_exercise': [0.0, 1.0, 0.0],   # one-hot: exercise
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}

print(f"Device: {CONFIG['device']}")

---
## Data Loading (WFDB Format)

The SensSmart-Tech database stores ECG (4 leads) and PCG as separate WFDB records.
Each record pair has suffixes `_ecg` and `_pcg`.
We select Lead II from the 4-channel ECG.
Native sampling rate is 1000 Hz; we resample to 8000 Hz via polyphase interpolation.

In [ ]:
def load_senssmarttech_record(record_base_name, wfdb_dir):
    """
    Load one ECG-PCG pair from the SensSmart-Tech WFDB directory.

    Parameters
    ----------
    record_base_name : str
        e.g. '1_09-30-00' (subject_timestamp)
    wfdb_dir : str
        Path to the WFDB directory

    Returns
    -------
    ecg : np.ndarray, shape (N,)  — Lead II
    pcg : np.ndarray, shape (N,)
    fs  : int  — original sampling rate (1000 Hz)
    """
    ecg_path = os.path.join(wfdb_dir, f"{record_base_name}_ecg")
    pcg_path = os.path.join(wfdb_dir, f"{record_base_name}_pcg")

    ecg_record = wfdb.rdrecord(ecg_path)
    pcg_record = wfdb.rdrecord(pcg_path)

    # Select Lead II from 4-channel ECG
    ecg_signals = ecg_record.p_signal        # (N, 4)
    lead_names = ecg_record.sig_name
    if CONFIG['ecg_lead'] in lead_names:
        lead_idx = lead_names.index(CONFIG['ecg_lead'])
    else:
        lead_idx = 1   # fallback: second channel

    ecg = ecg_signals[:, lead_idx].astype(np.float32)
    pcg = pcg_record.p_signal.squeeze().astype(np.float32)
    fs = ecg_record.fs

    return ecg, pcg, fs


def resample_to_target_fs(sig, fs_original, fs_target=8000):
    """
    Polyphase resampling from fs_original to fs_target.
    For 1000 -> 8000 Hz: up=8, down=1.
    """
    if fs_original == fs_target:
        return sig
    gcd = np.gcd(int(fs_target), int(fs_original))
    up = int(fs_target) // gcd
    down = int(fs_original) // gcd
    return resample_poly(sig, up, down).astype(np.float32)


def get_record_list(wfdb_dir):
    """
    Scan the WFDB directory for unique record base names.
    Records come in pairs: {name}_ecg and {name}_pcg.
    """
    files = os.listdir(wfdb_dir)
    ecg_headers = [f.replace('_ecg.hea', '')
                   for f in files if f.endswith('_ecg.hea')]
    return sorted(ecg_headers)


# Quick check
if os.path.exists(CONFIG['wfdb_dir']):
    records = get_record_list(CONFIG['wfdb_dir'])
    print(f"Found {len(records)} records in SensSmart-Tech WFDB directory")
else:
    print(f"WFDB directory not found: {CONFIG['wfdb_dir']}")
    print("Mount Google Drive or adjust CONFIG paths.")

---
## Preprocessing Pipeline

Matches the EPHNOGRAM preprocessing from Trail-4:
1. Bandpass filter (4th-order Butterworth)
2. Polarity correction (`if |min| > |max|: flip`)
3. Z-score normalization

ECG band: 0.5–40 Hz. PCG band: 25–400 Hz.

In [ ]:
def bandpass_filter(sig, fs, lowcut, highcut, order=4):
    """4th-order Butterworth bandpass filter (zero-phase)."""
    nyquist = fs / 2
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, sig).astype(np.float32)


def ensure_positive_polarity(ecg):
    """Flip ECG so R-peaks point upward (positive)."""
    if np.abs(np.min(ecg)) > np.abs(np.max(ecg)):
        ecg = -ecg
    return ecg


def normalize_zscore(sig):
    """Per-segment z-score normalization."""
    mu = sig.mean()
    std = sig.std() + 1e-8
    return ((sig - mu) / std).astype(np.float32)


def preprocess_smarttech(ecg_raw, pcg_raw, fs_original):
    """
    Full preprocessing pipeline for one SensSmart-Tech record.

    Steps:
        1. Resample 1000 -> 8000 Hz
        2. Bandpass filter
        3. Polarity correction (ECG only)
        4. Z-score normalization
        5. Truncate to segment_length (64000 samples = 8 s)

    Returns
    -------
    ecg, pcg : np.ndarray, shape (segment_length,)
    success  : bool
    """
    try:
        # Resample
        ecg = resample_to_target_fs(ecg_raw, fs_original, CONFIG['target_fs'])
        pcg = resample_to_target_fs(pcg_raw, fs_original, CONFIG['target_fs'])

        # Bandpass filter at target rate
        ecg = bandpass_filter(ecg, CONFIG['target_fs'],
                              CONFIG['ecg_bandpass'][0],
                              CONFIG['ecg_bandpass'][1],
                              CONFIG['filter_order'])
        pcg = bandpass_filter(pcg, CONFIG['target_fs'],
                              CONFIG['pcg_bandpass'][0],
                              CONFIG['pcg_bandpass'][1],
                              CONFIG['filter_order'])

        # Polarity correction
        if CONFIG['fix_polarity']:
            ecg = ensure_positive_polarity(ecg)

        # Z-score normalization
        ecg = normalize_zscore(ecg)
        pcg = normalize_zscore(pcg)

        # Truncate or pad to segment_length
        T = CONFIG['segment_length']
        if len(ecg) < T or len(pcg) < T:
            return None, None, False

        ecg = ecg[:T]
        pcg = pcg[:T]

        return ecg, pcg, True

    except Exception as e:
        print(f"  Preprocessing failed: {e}")
        return None, None, False


# Quick test with synthetic data
dummy = np.random.randn(30000).astype(np.float32)
ecg_out, pcg_out, ok = preprocess_smarttech(dummy, dummy, 1000)
if ok:
    print(f"Preprocessed: shape={ecg_out.shape}, "
          f"mean={ecg_out.mean():.4f}, std={ecg_out.std():.4f}")
else:
    print("Test signal too short after resampling (expected for 30s -> 8s)")

---
## Validation Loop

We load the Trail-4 generator, run inference on each preprocessed SmartTech record,
apply amplitude rescaling, and compute metrics.

**Scenario embedding:** `[0, 1, 0]` (exercise) for all SmartTech records.

**Amplitude rescaling:** The raw generator output has std ~ 0.277 vs. real std ~ 1.0.
We rescale: `fake_ecg = fake_ecg_raw * (real_std / fake_std)`.

In [ ]:
# --- Metrics (from metrics.ipynb) ---

def calc_psnr(real, fake):
    """PSNR (dB) with z-score signal range."""
    mse = np.mean((real - fake) ** 2)
    if mse < 1e-10:
        return 100.0
    peak = np.max(np.abs(real))
    return float(20 * np.log10(peak / np.sqrt(mse)))


def calc_ssim(real, fake):
    """SSIM after mapping both signals to [0, 1]."""
    rng_r = real.max() - real.min() + 1e-8
    rng_f = fake.max() - fake.min() + 1e-8
    r_n = (real - real.min()) / rng_r
    f_n = (fake - fake.min()) / rng_f
    return float(compare_ssim(r_n, f_n, data_range=1.0))


def calc_correlation(real, fake):
    """Pearson correlation coefficient."""
    try:
        c, _ = pearsonr(real.flatten(), fake.flatten())
        return float(c)
    except Exception:
        return 0.0


def detect_r_peaks(ecg, fs=8000, distance=400, prominence=0.5):
    """R-peak detection with scipy.signal.find_peaks."""
    peaks, props = signal.find_peaks(ecg, distance=distance,
                                     prominence=prominence)
    return peaks, props


def calc_r_peak_error(real, fake, fs=8000):
    """Mean absolute R-peak amplitude error."""
    peaks, _ = detect_r_peaks(real, fs=fs)
    if len(peaks) == 0:
        return 0.0
    return float(np.mean(np.abs(real[peaks] - fake[peaks])))


# --- Model loading ---
# The generator architecture is defined in models.ipynb.
# Copy or import ImprovedGeneratorNoTanh here, or load via torch.

def load_generator(model_path, device='cpu'):
    """
    Load the Trail-4 generator checkpoint.

    The checkpoint must contain the full model or state_dict
    for ImprovedGeneratorNoTanh (19.3M params).
    """
    checkpoint = torch.load(model_path, map_location=device)
    if isinstance(checkpoint, nn.Module):
        model = checkpoint
    else:
        # If state_dict, need the model class defined
        # Import from models.ipynb or define inline
        from models import ImprovedGeneratorNoTanh
        model = ImprovedGeneratorNoTanh()
        model.load_state_dict(checkpoint)

    model.eval()
    model.to(device)
    return model


# --- Validation ---

def validate_on_smarttech(generator, records_data, device='cpu'):
    """
    Run the generator on all preprocessed SmartTech records.

    Parameters
    ----------
    generator : nn.Module
    records_data : list of dict with 'ecg' and 'pcg' arrays
    device : str

    Returns
    -------
    results : list of dict (per-record metrics)
    summary : dict (aggregated statistics)
    """
    scenario = torch.FloatTensor(CONFIG['scenario_exercise']).unsqueeze(0).to(device)
    results = []
    std_ratios = []

    for i, rec in enumerate(records_data):
        pcg = rec['pcg']
        real_ecg = rec['ecg']

        # Model inference
        pcg_tensor = torch.FloatTensor(pcg).unsqueeze(0).unsqueeze(0).to(device)
        with torch.no_grad():
            fake_tensor = generator(pcg_tensor, scenario)
        fake_raw = fake_tensor.squeeze().cpu().numpy()

        # Amplitude diagnostics
        real_std = np.std(real_ecg)
        fake_std = np.std(fake_raw)
        std_ratio = fake_std / (real_std + 1e-8)
        std_ratios.append(std_ratio)

        # Amplitude rescaling
        fake_ecg = fake_raw * (real_std / (fake_std + 1e-8))

        # Metrics
        m = {
            'record': rec.get('name', f'rec_{i}'),
            'psnr': calc_psnr(real_ecg, fake_ecg),
            'ssim': calc_ssim(real_ecg, fake_ecg),
            'correlation': calc_correlation(real_ecg, fake_ecg),
            'r_peak_error': calc_r_peak_error(real_ecg, fake_ecg,
                                               fs=CONFIG['target_fs']),
            'std_ratio': std_ratio,
        }
        results.append(m)

        if (i + 1) % 50 == 0:
            print(f"  Processed {i+1}/{len(records_data)} records")

    # Aggregate
    metrics_keys = ['psnr', 'ssim', 'correlation', 'r_peak_error']
    summary = {}
    for k in metrics_keys:
        vals = [r[k] for r in results]
        summary[k] = {
            'mean': float(np.mean(vals)),
            'std': float(np.std(vals)),
            'min': float(np.min(vals)),
            'max': float(np.max(vals)),
            'median': float(np.median(vals)),
        }
    summary['amplitude_diagnostics'] = {
        'mean_real_std': float(np.mean([np.std(r['ecg'])
                                        for r in records_data])),
        'mean_fake_std_raw': float(np.mean([1.0 / (1.0/sr + 1e-8)
                                             for sr in std_ratios])),
        'mean_std_ratio': float(np.mean(std_ratios)),
    }
    summary['n_records'] = len(results)

    return results, summary

---
## Run Validation

Load preprocessed data, load model, and run the full validation loop.
Skip this cell if data is not available (results are pre-computed above).

In [ ]:
# --- Load preprocessed data ---
preprocessed_dir = CONFIG['preprocessed_dir']
records_data = []

if os.path.exists(preprocessed_dir):
    npz_files = sorted([f for f in os.listdir(preprocessed_dir)
                        if f.endswith('.npz')])
    print(f"Loading {len(npz_files)} preprocessed records...")

    for f in npz_files:
        data = np.load(os.path.join(preprocessed_dir, f))
        records_data.append({
            'name': f.replace('.npz', ''),
            'ecg': data['ecg'].astype(np.float32),
            'pcg': data['pcg'].astype(np.float32),
        })
    print(f"Loaded {len(records_data)} records")
else:
    print(f"Preprocessed directory not found: {preprocessed_dir}")
    print("Run preprocessing first or adjust paths.")

# --- Load model and validate ---
if records_data and os.path.exists(CONFIG['model_path']):
    generator = load_generator(CONFIG['model_path'],
                                device=CONFIG['device'])
    print(f"Generator loaded ({sum(p.numel() for p in generator.parameters()):,} params)")

    results, summary = validate_on_smarttech(
        generator, records_data, device=CONFIG['device'])

    # Save results
    os.makedirs(CONFIG['results_dir'], exist_ok=True)
    with open(os.path.join(CONFIG['results_dir'],
                           'validation_summary.json'), 'w') as f:
        json.dump(summary, f, indent=2)
    print(f"\nResults saved to {CONFIG['results_dir']}")
else:
    print("Skipping validation (data or model not found)")
    # Pre-computed results from Milestone-2
    summary = {
        'psnr': {'mean': 17.035, 'std': 8.939},
        'ssim': {'mean': 0.742, 'std': 0.099},
        'correlation': {'mean': -0.003, 'std': 0.129},
        'r_peak_error': {'mean': 1.400, 'std': 0.439},
        'n_records': 310,
    }

---
## Results Comparison

EPHNOGRAM (training domain) vs. SensSmart-Tech (unseen domain).

In [ ]:
# EPHNOGRAM Trail-4 test-set results
ephnogram = {
    'psnr': 17.10,
    'ssim': 0.903,
    'correlation': 0.610,
    'r_peak_error': 0.126,
    'n_records': 453,
}

# SmartTech results (from validation or pre-computed)
smarttech = {
    'psnr': summary['psnr']['mean'],
    'ssim': summary['ssim']['mean'],
    'correlation': summary['correlation']['mean'],
    'r_peak_error': summary['r_peak_error']['mean'],
    'n_records': summary.get('n_records', 310),
}

print("=" * 65)
print("Cross-Dataset Validation: EPHNOGRAM vs. SensSmart-Tech")
print("=" * 65)
print(f"{'Metric':<20s}  {'EPHNOGRAM':>10s}  {'SmartTech':>10s}  {'Delta':>10s}")
print("-" * 65)

for metric, label in [('psnr', 'PSNR (dB)'),
                       ('ssim', 'SSIM'),
                       ('correlation', 'Correlation'),
                       ('r_peak_error', 'R-peak Err')]:
    e = ephnogram[metric]
    s = smarttech[metric]
    if e != 0:
        delta = (s - e) / abs(e) * 100
    else:
        delta = 0
    print(f"{label:<20s}  {e:>10.3f}  {s:>10.3f}  {delta:>+9.1f}%")

print("-" * 65)
print(f"{'Records':<20s}  {ephnogram['n_records']:>10d}  "
      f"{smarttech['n_records']:>10d}")
print()
print("PSNR matches within 0.4%: amplitude reconstruction transfers.")
print("Correlation collapses: temporal alignment does not transfer.")
print("Domain adaptation is needed for deployment on new hardware.")